# Руководство по SimulacraBench

Мы воспользуемся учебной схемой
`data/sample.json`, чтобы понять форму задачи и более техническую причину проведения конкурса. Контракт подачи, правило подсчёта очков и регламент — в `README.md`.

In [1]:
import os
import sys
from pathlib import Path

# Этот блокнот лежит в tutorials/, но код лежит в корне
# репозитория, и все пути ниже -- config.yml, data/sample.json, _sandbox/ --
# записаны относительно этого корня. Находим его и работаем оттуда, чтобы
# блокнот вёл себя одинаково независимо от того, запущен ли Jupyter в этой
# папке или уровнем выше.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "make_sandbox.py").exists()), None)
if ROOT is None:
    raise RuntimeError("запускайте этот блокнот внутри клона репозитория")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

# Те же модули, что использует score.py. Здесь ничто не
# переписывает проверяющую программу: оценивая что-либо в этом блокноте, вы
# вызываете тот же код, который оценивает вас.
from make_sandbox import (ROLE_COLUMN, generated_items, load_config,
                          load_schema, options_for, write_sandbox)
from score import floored, load_frames, sample_rows
from score import score as grade

SEED = 0
PHASE = 1
config = load_config("config.yml")

pd.set_option("display.width", 200, "display.max_columns", 50)

---

## 1. Форма задачи

`make_sandbox.py` превращает схему в набор данных ровно той же формы, что и
настоящий: те же столбцы, те же варианты ответа, та же логика переходов.
Маргинальные распределения и зависимости вымышлены, поэтому отлаженный здесь
конвейер перенесётся, а настроенная здесь модель — нет.

Он записывает `respondents.parquet` — каждого респондента и роль `role`,
указывающую, для чего он нужен, — и `schema.json`, то самое, что получает ваш
`predict()`. Роли назначаются один раз, здесь, и записываются на диск.
`score.py` их только читает; сам он ничего не делит.

In [2]:
sample = load_schema("data/sample.json", config)
write_sandbox(sample, config, "_sandbox/sample", seed=SEED)

respondents = load_frames("_sandbox/sample", sample)
print(sample["dataset"]["description"])
print()

MEANING = {"TRAIN": "видим в обеих фазах",
           "DEV": "оценивается в фазе 1, видим в фазе 2",
           "FINAL": "оценивается в фазе 2"}
counts = respondents[ROLE_COLUMN].value_counts()
print(pd.DataFrame({"респондентов": counts,
                    "значение": [MEANING[r] for r in counts.index]}).to_string())

A toy instrument, not a real survey. Ten items, few enough to print the whole schema and read it. It has one of everything the real schemas have: a frame block that is always visible, items that are scored, a gate chain two deep, and an EXCLUDE column the grader never shows anybody. The GIVEN block is deliberately the cheap half of a questionnaire -- the variables that already sit on a sampling frame, a census roster or another survey of the same households -- and the PREDICT block is the expensive half, the part that needs an enumerator and an interview. Use it to see the shape of the task; use the three real schemas to see whether a method works.

       респондентов                              значение
role                                                     
TRAIN          8981                   видим в обеих фазах
FINAL          2112                  оценивается в фазе 2
DEV             907  оценивается в фазе 1, видим в фазе 2


### Что объявляет схема

У каждого пункта четыре ключа: `question` — формулировка вопроса, `class`,
`values` — допустимые ответы, и `gate`, если пункт задаётся не всем.

Три класса и составляют всю задачу. **`GIVEN`** виден всем и никогда не
оценивается. **`PREDICT`** скрыт у скрытых респондентов, и каждая пустая
ячейка оценивается. **`EXCLUDE`** — идентификаторы, ключи записей, свободный
текст — вообще не передаётся в таблице, поэтому фильтруйте по `class`, а не
предполагайте, что схема и таблица содержат одни и те же столбцы.

В этом инструменте разделение намеренно противопоставляет дешёвую половину
анкеты дорогой. Блок `GIVEN` — это то, что уже есть в основе выборки, в
переписи или в другом обследовании тех же домохозяйств: где человек живёт,
каков размер домохозяйства, есть ли телефон. Блок `PREDICT` — это то, что
требует интервьюера и беседы. На этом различии построена часть 2.

In [3]:
rows = []
for name, rec in sample["items"].items():
    gate = rec.get("gate") or {}
    rows.append({"пункт": name,
                 "класс": rec["class"],
                 "K": len(options_for(sample, name)) if rec["values"] else 0,
                 "переход": gate.get("parent", "-"),
                 "задан если": ", ".join(gate.get("observed_if", [])) or "-",
                 "варианты": " | ".join(map(str, rec["values"] or ["-"]))})
print(pd.DataFrame(rows).to_string(index=False))

                 пункт   класс  K        переход                                            задан если                                                варианты
                region   GIVEN  3              -                                                     -                                 North | Central | South
           urban_rural   GIVEN  2              -                                                     -                                           Urban | Rural
              age_band   GIVEN  4              -                                                     -                             18-29 | 30-44 | 45-59 | 60+
        household_size   GIVEN  4              -                                                     -                               1 | 2-3 | 4-5 | 6 or more
household_has_children   GIVEN  2              -                                                     -                                                Yes | No
      has_mobile_phone   GIVEN  2             

`K` — число вариантов пункта **вместе со служебным значением перехода**: у
пункта с переходом на одну ячейку больше, чем ответов, потому что «не
задавался» для него полноценный ответ. Эта дополнительная ячейка — последняя в
вашем векторе вероятностей, и именно из `K` вычисляется равномерный эталон `U`.

`would_return` зависит от `clinic_wait`, а тот — от `visited_clinic`: цепочка
глубиной два. Тому, кто не обращался в поликлинику, не задавали ни вопрос о
времени ожидания, ни вопрос о повторном визите. Истинный ответ на оба —
`NA_GATED`.

In [4]:
chain = ["visited_clinic", "clinic_wait", "would_return"]
print(respondents[chain].value_counts().to_frame("респондентов").head(12).to_string())

                                                         респондентов
visited_clinic       clinic_wait           would_return              
No                   NA_GATED              NA_GATED              5179
Prefer not to answer NA_GATED              NA_GATED              4413
Yes                  Over 2 hours          Yes                   1405
                     Under 30 minutes      Yes                    380
                     Over 2 hours          No                     204
                                           Not sure               201
                     Under 30 minutes      Not sure               162
                                           No                      28
                     30 minutes to 2 hours Yes                     21
                                           Not sure                 4
                                           No                       3


Читайте эту таблицу как саму логику переходов: везде, где `visited_clinic` не
равен `Yes`, оба дочерних пункта равны `NA_GATED`, без исключений. **Ответ
пункта с переходом определён, как только виден его родитель** — это бесплатные
очки и первое, чем стоит воспользоваться.

### Что получает `predict()`

`score.py` берёт видимые и скрытые роли этой фазы, ставит видимых респондентов
над скрытыми и очищает каждую ячейку `PREDICT` у последних. Именно для этих
пустых ячеек вы возвращаете вероятности.

`NaN` означает ровно одно: *эта ячейка скрыта, предскажите её*. Он никогда не
означает «человек не ответил» — настоящий отказ от ответа является обычным
вариантом вроде `Prefer not to answer` и стоит в списке вариантов наравне с
остальными.

In [5]:
frame, cells, truth = sample_rows(sample, respondents, config, PHASE, seed=SEED)
shown = [n for n, r in sample["items"].items() if r["class"] != "EXCLUDE"]

print("таблица:", frame.shape, " ячеек для прогноза:", len(cells))
print()
print(pd.concat([frame[["respondent_id"] + shown].head(3),
                 frame[["respondent_id"] + shown].tail(3)]).to_string(index=False))

таблица: (3564, 11)  ячеек для прогноза: 3628

respondent_id  region urban_rural age_band household_size household_has_children has_mobile_phone visited_clinic clinic_wait would_return trusts_health_advice
      R000002   North       Rural      60+            4-5                    Yes               No             No    NA_GATED     NA_GATED           Not at all
      R000003   South       Rural      60+            2-3                     No               No             No    NA_GATED     NA_GATED                A lot
      R000005   South       Rural    18-29              1                    Yes               No             No    NA_GATED     NA_GATED                A lot
      R011985 Central       Urban    45-59            4-5                     No               No            NaN         NaN          NaN                  NaN
      R011987   South       Rural      60+            2-3                    Yes               No            NaN         NaN          NaN                  NaN

Верхние строки — видимые респонденты: заполненные, и на них можно учиться.
Нижние строки скрыты — вы видите их блок `GIVEN` и больше ничего.

Возвращаемое значение — по одному вектору вероятностей на пустую ячейку, в
**каноническом порядке**: строки сверху вниз, а внутри строки — пункты в
порядке ключей `schema["items"]`, а не в порядке `frame.columns`, который может
отличаться. Каждый вектор следует за `values` пункта по порядку, плюс служебная
ячейка, если у пункта есть переход. Читайте порядок из схемы, а не из данных:
вариант, который никто не выбрал, всё равно занимает ячейку.

In [6]:
print(pd.DataFrame(cells, columns=["строка", "respondent_id", "пункт"]).head(8)
      .to_string(index=False))

 строка respondent_id                пункт
   2657       R000011       visited_clinic
   2657       R000011          clinic_wait
   2657       R000011         would_return
   2657       R000011 trusts_health_advice
   2658       R000022       visited_clinic
   2658       R000022          clinic_wait
   2658       R000022         would_return
   2658       R000022 trusts_health_advice


### Подсчёт очков

Базовое решение «толпы»: каждый скрытый респондент получает сглаженные доли по
каждому пункту, без всякого учёта личности. `skill` равен 0 для равномерного
ответа и 1 для совершенного, и именно по нему строится таблица результатов.

In [7]:
def hidden_cells(frame, items):
    '''Каждая пустая ячейка в том порядке, в каком predict() должен их вернуть.'''
    values = frame[items].to_numpy(dtype=object)
    ids = frame["respondent_id"].to_numpy(dtype=object)
    return [(row, ids[row], items[col])
            for row in range(values.shape[0])
            for col in range(len(items))
            if pd.isna(values[row, col])]


def crowd_for(sch, frame):
    items = generated_items(sch)
    tables = {}
    for item in items:
        counts = frame[item].value_counts()
        n = np.array([counts.get(o, 0) for o in options_for(sch, item)], float)
        tables[item] = (n + 0.5) / (n + 0.5).sum()
    return [tables[item] for _, _, item in hidden_cells(frame, items)]


vectors = floored(crowd_for(sample, frame), sample, cells, config["scoring"]["floor"])
result = grade(sample, config, vectors, truth, cells)

def table(rows):
    """Print label/value pairs, aligned however long the labels happen to be."""
    pad = max(len(label) for label, _ in rows)
    for label, value in rows:
        print("%-*s  %s" % (pad, label, value))


table([("равномерный эталон (наты)", "%.4f" % result["uniform_reference"]),
       ("лог-оценка", "%.4f" % result["log_score"]),
       ("skill", "%.4f" % result["skill"])])
print()
print("skill равен 0 для равномерного ответа и 1 для совершенного.")

равномерный эталон (наты)  1.3144
лог-оценка                 -0.8941
skill                      0.3198

skill равен 0 для равномерного ответа и 1 для совершенного.


Вот и весь контракт. Подача — это `main.py` с функцией `predict()`,
возвращающей такие векторы; `score.py` запускает её так же, как это сделает
проверяющая программа, а `tools/check_submission_zip.py` проверяет, что
загружаемый архив составлен правильно.

---

## 2. Что даёт хорошая модель

Бенчмарк, поощряющий предсказание ответов людей, вызывает очевидное
опасение: не в том ли смысл, чтобы перестать их спрашивать? Здесь мы увидим способ сочетать алгоритмические прогнозы с человеческими выборками.

Нам нужно одно число о населении: доля домохозяйств, доверяющих советам своей поликлиники по здоровью. Дешёвый блок — регион, город или село,
размер домохозяйства, наличие телефона — уже известен для каждого домохозяйства
в основе выборки из административных данных или прошлого обследования. Дорогой
блок требует интервьюера у двери, а бюджета хватает на несколько сотен бесед.

У вас три варианта.

1. **Только интервью.** Опросить 300 домохозяйств, взять долю, опубликовать
   доверительный интервал. Корректно и настолько точно, насколько позволяют
   300 бесед.
2. **Только модель.** Прогнать модель по дешёвому блоку каждого домохозяйства
   и опубликовать среднее. Бесплатно и *ошибочно ровно настолько, насколько
   ошибается модель* — без интервала и без способа это узнать.
3. **И то и другое.** Использовать модель везде, а затем по 300 беседам
   измерить и вычесть ошибку модели. Это **вывод с опорой на прогноз**, и
   именно его строит остаток раздела.

Третий вариант и есть искомый: он корректен независимо от того, хороша модель
или плоха, и *точнее первого, когда модель хороша*.

In [8]:
# Оцениваемая величина: доля тех, кто хотя бы отчасти доверяет советам по здоровью.
TARGET, POSITIVE = "trusts_health_advice", ["Somewhat", "A lot"]
GIVEN = [n for n, r in sample["items"].items() if r["class"] == "GIVEN"]

Y = respondents[TARGET].isin(POSITIVE).to_numpy(float)
design = pd.get_dummies(respondents[GIVEN].astype(str), drop_first=True)
X = np.column_stack([np.ones(len(design)), design.to_numpy(float)])

# Модель обучена на респондентах прошлых волн -- роль TRAIN,
# то есть ровно тот видимый блок, на котором учится подача. Домохозяйства,
# которые мы собираемся опросить, она не видит никогда.
past = (respondents[ROLE_COLUMN] == "TRAIN").to_numpy()
ridge = np.linalg.solve(X[past].T @ X[past] + 5 * np.eye(X.shape[1]),
                        X[past].T @ Y[past])
predicted = X @ ridge          # f(дешёвый блок) для каждого домохозяйства

# Совокупность, о которой нужно число: домохозяйства, не использованные для обучения.
frame_rows = np.flatnonzero(~past)
TRUTH = Y[frame_rows].mean()   # известно лишь потому, что данные вымышлены

table([("модель обучена на, респондентов прошлых волн:", "%d" % past.sum()),
       ("оцениваемая совокупность, домохозяйств:", "%d" % len(frame_rows)),
       ("корреляция между прогнозом и ответом:", "%.2f"
        % np.corrcoef(predicted[frame_rows], Y[frame_rows])[0, 1]),
       ("истинная доля (которую настоящее обследование не видит):", "%.3f" % TRUTH)])

модель обучена на, респондентов прошлых волн:             8981
оцениваемая совокупность, домохозяйств:                   3019
корреляция между прогнозом и ответом:                     0.57
истинная доля (которую настоящее обследование не видит):  0.597


Теперь отберём 300 бесед и посчитаем все три числа.

Интервал «только интервью» — учебниковый. Интервал с опорой на прогноз — это
среднее модели по домохозяйствам, которые вы **не** опрашивали, поправленное
на среднюю ошибку модели по тем, кого опрашивали:

```
оценка = среднее(прогноз | не опрошены) - [ среднее(прогноз | опрошены) - среднее(ответ | опрошены) ]
                ↑ модель, применённая везде        ↑ измеренная ошибка модели
```

Эта скобка и есть весь предохранительный механизм. Она считается по реальным
ответам, то есть стоит реальных бесед, и устраняет смещение модели, каким бы
оно ни было.

In [9]:
def estimates(f, interviewed, rest):
    '''Оценки «только интервью» и с опорой на прогноз, каждая со стандартной ошибкой.'''
    y = Y[interviewed]
    classical = (y.mean(), y.std(ddof=1) / np.sqrt(len(y)))

    correction = f[interviewed].mean() - y.mean()
    powered = (f[rest].mean() - correction,
               np.sqrt(f[rest].var(ddof=1) / len(rest)
                       + (f[interviewed] - y).var(ddof=1) / len(interviewed)))
    return classical, powered


def band(estimate):
    point, se = estimate
    return "%.3f  [%.3f, %.3f]  ширина %.3f" % (
        point, point - 1.96 * se, point + 1.96 * se, 2 * 1.96 * se)


N_INTERVIEWS = 300
draw = np.random.default_rng(SEED).permutation(frame_rows)
interviewed, rest = draw[:N_INTERVIEWS], draw[N_INTERVIEWS:]

classical, powered = estimates(predicted, interviewed, rest)
table([("истина", "%.3f" % TRUTH),
       ("только интервью", band(classical)),
       ("с опорой на прогноз", band(powered)),
       ("только модель (без интервью)", "%.3f  [интервала нет вовсе]"
        % predicted[frame_rows].mean())])

истина                        0.597
только интервью               0.620  [0.565, 0.675]  ширина 0.110
с опорой на прогноз           0.609  [0.560, 0.658]  ширина 0.099
только модель (без интервью)  0.596  [интервала нет вовсе]


Одна выборка ничего не доказывает — интервалу могло повезти. Важно поведение на
множестве обследований: содержит ли интервал истину примерно в 95 % случаев и
какова его ширина? Повторим весь опыт тысячу раз, каждый раз с новыми 300
домохозяйствами.

In [10]:
def repeat(f, n_interviews=N_INTERVIEWS, draws=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(draws):
        shuffled = rng.permutation(frame_rows)
        classical, powered = estimates(f, shuffled[:n_interviews],
                                       shuffled[n_interviews:])
        out.append(classical + powered)
    return np.array(out)          # оценка, стандартная ошибка, оценка, стандартная ошибка


def summarize(trials, label):
    rows = []
    for name, point, se in (("только интервью", trials[:, 0], trials[:, 1]),
                            ("с опорой на прогноз", trials[:, 2], trials[:, 3])):
        rows.append({"метод": name,
                     "средняя ширина": (2 * 1.96 * se).mean(),
                     "содержит истину": np.mean(np.abs(point - TRUTH) <= 1.96 * se)})
    out = pd.DataFrame(rows)
    print(label)
    print(out.to_string(index=False, float_format="%.3f"))
    return out


good = summarize(repeat(predicted), "модель, которая хорошо предсказывает")
narrower = 1 - good.loc[1, "средняя ширина"] / good.loc[0, "средняя ширина"]
print("\nполоса на %.0f %% уже при тех же %d беседах." % (100 * narrower, N_INTERVIEWS))
print("чтобы купить такую точность одними беседами, их понадобилось бы около %d." % round(N_INTERVIEWS / (1 - narrower) ** 2))

модель, которая хорошо предсказывает
              метод  средняя ширина  содержит истину
    только интервью           0.111            0.961
с опорой на прогноз           0.093            0.964

полоса на 16 % уже при тех же 300 беседах.
чтобы купить такую точность одними беседами, их понадобилось бы около 424.


Оба интервала содержат истину примерно в 95 % случаев — это и делает их
интервалами. Интервал с опорой на прогноз просто **уже**, при ровно той же
полевой работе. Последнюю строку читайте как смысл всего упражнения: лучшая
модель не убирает беседы из бюджета, она делает каждую весомее.

### Что происходит, когда модель плоха

Очевидное возражение: это работает, лишь пока модель права, и довериться ей и
есть риск. Вот та же процедура с моделью, обученной на **другом населении**.

In [11]:
from make_sandbox import make_sandbox

elsewhere = make_sandbox(sample, config, seed=99)      # другое население
other_design = pd.get_dummies(elsewhere[GIVEN].astype(str), drop_first=True)
other_X = np.column_stack([np.ones(len(other_design)), other_design.to_numpy(float)])
other_Y = elsewhere[TARGET].isin(POSITIVE).to_numpy(float)

wrong = np.linalg.solve(other_X.T @ other_X + 5 * np.eye(other_X.shape[1]),
                        other_X.T @ other_Y)
mispredicted = X @ wrong

table([("корреляция между прогнозом и ответом:", "%.2f"
        % np.corrcoef(mispredicted[frame_rows], Y[frame_rows])[0, 1]),
       ("только модель (без интервью)", "%.3f   против истины %.3f   <- отклонение %+.3f"
        % (mispredicted[frame_rows].mean(), TRUTH,
           mispredicted[frame_rows].mean() - TRUTH))])
print()
summarize(repeat(mispredicted), "модель, которая не переносится")

корреляция между прогнозом и ответом:  0.09


только модель (без интервью)           0.725   против истины 0.597   <- отклонение +0.129



модель, которая не переносится
              метод  средняя ширина  содержит истину
    только интервью           0.111            0.961
с опорой на прогноз           0.116            0.960


,метод,средняя ширина,содержит истину
0,только интервью,0.111063,0.961
1,с опорой на прогноз,0.116003,0.960


Обратите внимание: оценка «только модель» ошибается больше чем на одну десятую,
и ничто в выводе вам об этом не сообщило бы — ни интервала, ни предупреждения,
просто число, выглядящее ровно так же авторитетно, как верное. Замена поля
моделью вносит смещение.

Интервал с опорой на прогноз по-прежнему содержит истину примерно в 95 %
случаев. Он не уже, чем при одних беседах — бесполезная модель не покупает
точности. Поправочный член измерил ошибку модели на 300 реальных беседах и
вычел её, ровно для этого он и нужен.

### Зачем для этого бенчмарк

Ширина этой полосы — прямая функция качества модели. Отсюда и смысл тщательно
измерять качество прогноза на реальных инструментах строгим правилом подсчёта
очков: более высокий `skill` в таблице результатов — это более узкий
доверительный интервал в поле или тот же интервал при меньшем числе бесед.